# Row Filtering and Column Masking latest features.


# Sample data and table
### Prompt : Generate pyspark code to create a Unity Catalog Delta table with PII information and 50 records between 5 regions of USA. Inclue First Name, Last Name, Date of Birth, Age, Sex, Address, SSN and geographic region columns.


In [0]:
%sql
drop schema if exists dkushari_uc.fgac cascade;
create schema if not exists dkushari_uc.fgac;

In [0]:
%sql
use catalog dkushari_uc;
use fgac;

In [0]:
%sql
select current_catalog();

current_catalog()
dkushari_uc


In [0]:
%sql
select current_schema();

current_schema()
fgac


In [0]:
%sql 
DROP TABLE if exists customer_pii_data;

In [0]:
from pyspark.sql.functions import expr, lit, rand
from pyspark.sql.types import IntegerType

# Generate a DataFrame with 50 records
df = spark.range(50).withColumnRenamed("id", "record_id")

# Add PII information (dummy data for illustration)
df = df.withColumn("first_name", expr("CASE WHEN rand() < 0.2 THEN 'John' " +
                                      "WHEN rand() < 0.4 THEN 'Jane' " +
                                      "WHEN rand() < 0.6 THEN 'Doe' " +
                                      "WHEN rand() < 0.8 THEN 'Alice' " +
                                      "ELSE 'Bob' END")) \
      .withColumn("last_name", expr("CASE WHEN rand() < 0.2 THEN 'Smith' " +
                                    "WHEN rand() < 0.4 THEN 'Johnson' " +
                                    "WHEN rand() < 0.6 THEN 'Williams' " +
                                    "WHEN rand() < 0.8 THEN 'Brown' " +
                                    "ELSE 'Jones' END")) \
      .withColumn("date_of_birth", expr("CASE WHEN rand() < 0.2 THEN '1980-01-01' " +
                                        "WHEN rand() < 0.4 THEN '1990-01-01' " +
                                        "WHEN rand() < 0.6 THEN '2000-01-01' " +
                                        "WHEN rand() < 0.8 THEN '1985-01-01' " +
                                        "ELSE '1995-01-01' END")) \
      .withColumn("age", (lit(2023) - expr("substring(date_of_birth, 1, 4)")).cast(IntegerType())) \
      .withColumn("sex", expr("CASE WHEN rand() < 0.5 THEN 'M' ELSE 'F' END")) \
      .withColumn("address", expr("CASE WHEN rand() < 0.2 THEN '123 Main St' " +
                                  "WHEN rand() < 0.4 THEN '456 Elm St' " +
                                  "WHEN rand() < 0.6 THEN '789 Pine St' " +
                                  "WHEN rand() < 0.8 THEN '101 Oak St' " +
                                  "ELSE '202 Maple St' END")) \
      .withColumn("ssn", expr("CASE WHEN rand() < 0.2 THEN '111-11-1111' " +
                              "WHEN rand() < 0.4 THEN '222-22-2222' " +
                              "WHEN rand() < 0.6 THEN '333-33-3333' " +
                              "WHEN rand() < 0.8 THEN '444-44-4444' " +
                              "ELSE '555-55-5555' END")) \
      .withColumn("region", expr("CASE WHEN rand() < 0.2 THEN 'Northeast' " +
                                 "WHEN rand() < 0.4 THEN 'Midwest' " +
                                 "WHEN rand() < 0.6 THEN 'South' " +
                                 "WHEN rand() < 0.8 THEN 'West' " +
                                 "ELSE 'Southwest' END"))

# Display the DataFrame
display(df)

# Write the DataFrame to a Unity Catalog Delta table
df.write.format("delta").mode("overwrite").saveAsTable("customer_pii_data")

record_id,first_name,last_name,date_of_birth,age,sex,address,ssn,region
0,Jane,Brown,1980-01-01,43,M,789 Pine St,222-22-2222,West
1,Doe,Williams,2000-01-01,23,M,456 Elm St,222-22-2222,Northeast
2,John,Johnson,2000-01-01,23,F,789 Pine St,222-22-2222,West
3,Doe,Johnson,1990-01-01,33,F,456 Elm St,333-33-3333,South
4,John,Williams,2000-01-01,23,F,123 Main St,333-33-3333,South
5,Jane,Smith,1980-01-01,43,M,123 Main St,111-11-1111,South
6,Jane,Williams,2000-01-01,23,F,789 Pine St,222-22-2222,South
7,Jane,Jones,1980-01-01,43,F,101 Oak St,222-22-2222,Midwest
8,Jane,Smith,2000-01-01,23,M,456 Elm St,222-22-2222,South
9,John,Smith,1980-01-01,43,M,123 Main St,222-22-2222,Northeast


In [0]:
%sql
delete from dkushari_uc.fgac.customer_pii_data where first_name = 'Doe' and last_name = 'Williams'

num_affected_rows
5


In [0]:
%sql
insert into dkushari_uc.fgac.customer_pii_data
select * from dkushari_uc.fgac.customer_pii_data limit 100;

num_affected_rows,num_inserted_rows
45,45


In [0]:
%sql
update dkushari_uc.fgac.customer_pii_data set first_name = 'John'
where first_name = 'Doe' 

num_affected_rows
18


In [0]:
%sql
insert into dkushari_uc.fgac.customer_pii_data
select * from dkushari_uc.fgac.customer_pii_data where first_name = 'Doe' and last_name = 'Williams'

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
desc history dkushari_uc.fgac.customer_pii_data;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-03-20T19:49:04.000Z,6235569048193142,dipankar.kushari@databricks.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2932186303220612),db3000ff-cae7-43b3-bb2a-dd7c3e7850c5,0320-150033-6p5vmoom-v2n,5,WriteSerializable,false,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/18.0.x-photon-scala2.13
5,2026-03-20T19:49:00.000Z,6235569048193142,dipankar.kushari@databricks.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2932186303220612),5828bb97-3a4b-46ab-a71c-ed67baf1d361,0320-150033-6p5vmoom-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 9181, p25FileSize -> 3345, numDeletionVectorsRemoved -> 2, minFileSize -> 3345, numAddedFiles -> 1, maxFileSize -> 3345, p75FileSize -> 3345, p50FileSize -> 3345, numAddedBytes -> 3345)",null,Databricks-Runtime/18.0.x-photon-scala2.13
4,2026-03-20T19:48:59.000Z,6235569048193142,dipankar.kushari@databricks.com,UPDATE,"Map(predicate -> [""(first_name#109990 = Doe)""])",null,List(2932186303220612),5828bb97-3a4b-46ab-a71c-ed67baf1d361,0320-150033-6p5vmoom-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1912, numDeletionVectorsUpdated -> 0, scanTimeMs -> 792, numAddedFiles -> 1, numUpdatedRows -> 18, numAddedBytes -> 2913, rewriteTimeMs -> 1119)",null,Databricks-Runtime/18.0.x-photon-scala2.13
3,2026-03-20T19:48:55.000Z,6235569048193142,dipankar.kushari@databricks.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2932186303220612),f6528746-2f0a-479e-a914-d46beb68c67e,0320-150033-6p5vmoom-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 45, numOutputBytes -> 3132, conflictDetectionTimeMs -> 59)",null,Databricks-Runtime/18.0.x-photon-scala2.13
2,2026-03-20T19:48:54.000Z,6235569048193142,dipankar.kushari@databricks.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2932186303220612),878d05af-646d-49fe-b144-203dd7485e75,0320-150033-6p5vmoom-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 8, numRemovedBytes -> 22267, p25FileSize -> 3136, numDeletionVectorsRemoved -> 4, minFileSize -> 3136, numAddedFiles -> 1, maxFileSize -> 3136, p75FileSize -> 3136, p50FileSize -> 3136, numAddedBytes -> 3136)",null,Databricks-Runtime/18.0.x-photon-scala2.13
1,2026-03-20T19:48:52.000Z,6235569048193142,dipankar.kushari@databricks.com,DELETE,"Map(predicate -> [""((first_name#109385 = Doe) AND (last_name#109386 = Williams))""])",null,List(2932186303220612),878d05af-646d-49fe-b144-203dd7485e75,0320-150033-6p5vmoom-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 4, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1519, numDeletionVectorsUpdated -> 0, numDeletedRows -> 5, scanTimeMs -> 1219, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 298)",null,Databricks-Runtime/18.0.x-photon-scala2.13
0,2026-03-20T19:48:45.000Z,6235569048193142,dipankar.kushari@databricks.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2932186303220612),fbb36150-02f6-44e3-b8c5-8891b24c1c65,0320-150033-6p5vmoom-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50, numOutputBytes -> 22267)",null,Databricks-Runtime/18.0.x-photon-scala2

## Customer Table with PII data

In [0]:
%sql
SELECT region, COUNT(*) AS total_customers
FROM customer_pii_data
GROUP BY region;

region,total_customers
South,26
Midwest,24
Northeast,26
West,14


# UC Group membership lookup

In [0]:
%sql
SHOW GROUPS WITH USER `dipankar.kushari@databricks.com`;

name,directGroup
team.fieldeng-all,true
account users,true
admins,true
dkushari-test-group-2025,true
analyst_usa_tta,true
vibe-coding,true
demo_fgac_group_2,true
mars_group_2,true


In [0]:
%sql
DROP TABLE IF EXISTS dkushari_uc.fgac.valid_users;

CREATE TABLE dkushari_uc.fgac.valid_users(username string);
INSERT INTO dkushari_uc.fgac.valid_users
VALUES
  ('dipankar.kushari@databricks.com'),
  ('sreeramreddy.thoom@databricks.com');


num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
drop table if exists valid_accounts;
CREATE TABLE if not exists valid_accounts(account string);
INSERT INTO valid_accounts
VALUES
  ('ANALYST_ES'),
  ('ANALYST_SPAIN');



num_affected_rows,num_inserted_rows
2,2


# Create row filter and column masking UC functions 

age is a variable / 30 is also variable

column name and the value both can change

In [0]:
%sql
drop function if exists customer_region_filter;
CREATE OR REPLACE FUNCTION customer_region_filter(region STRING, age int)
  RETURN IF ( is_account_group_member("ANALYST_SPAIN"), TRUE,  region="West" 
  and age > 30 
  and EXISTS(SELECT 1 FROM valid_users v
    WHERE v.username = CURRENT_USER()))
  OR EXISTS(
    SELECT 1 FROM valid_accounts v
    WHERE IS_ACCOUNT_GROUP_MEMBER(v.account));

In [0]:
%sql
drop function if exists customer_ssn_mask;
CREATE OR REPLACE FUNCTION customer_ssn_mask(ssn STRING) 
  RETURN IF(is_account_group_member("ANALYST_SPAIN"), ssn, "+_)(*&^%$#@!)");


## Alter table to add row filter and column mask

In [0]:
%sql
ALTER TABLE customer_pii_data
  SET ROW FILTER customer_region_filter ON (region, age);

In [0]:
%sql
ALTER TABLE customer_pii_data
  ALTER COLUMN ssn SET MASK customer_ssn_mask;

In [0]:
%sql
select * from customer_pii_data order by record_id;

record_id,first_name,last_name,date_of_birth,age,sex,address,ssn,region
0,Jane,Brown,1980-01-01,43,M,789 Pine St,+_)(*&^%$#@!),West
0,Jane,Brown,1980-01-01,43,M,789 Pine St,+_)(*&^%$#@!),West
21,Jane,Williams,1980-01-01,43,M,101 Oak St,+_)(*&^%$#@!),West
21,Jane,Williams,1980-01-01,43,M,101 Oak St,+_)(*&^%$#@!),West
25,Jane,Williams,1990-01-01,33,M,456 Elm St,+_)(*&^%$#@!),West
25,Jane,Williams,1990-01-01,33,M,456 Elm St,+_)(*&^%$#@!),West
28,John,Smith,1980-01-01,43,F,789 Pine St,+_)(*&^%$#@!),West
28,John,Smith,1980-01-01,43,F,789 Pine St,+_)(*&^%$#@!),West
30,John,Williams,1990-01-01,33,M,456 Elm St,+_)(*&^%$#@!),West
30,John,Williams,1990-01-01,33,M,456 Elm St,+_)(*&^%$#@!),West


In [0]:
%sql
SELECT region, COUNT(*) AS total_customers
FROM customer_pii_data
GROUP BY region;

region,total_customers
West,12


In [0]:
%sql
select count(*) from customer_pii_data;

count(*)
12


In [0]:
# version: 1.1

# source: dkushari_uc.fgac.customer_pii_data

# dimensions:
#   - name: age
#     expr: source.age
#     display_name: Age
#   - name: sex
#     expr: source.sex
#     display_name: Sex
#   - name: region
#     expr: source.region
#     display_name: Region

# measures:
#   - name: count
#     expr: COUNT(*)
#     comment: Represents the total number of rows in the dataset. Use this measure
#       to count all
#     display_name: Count

In [0]:
%sql
select region, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by region

region,measure(count)
West,12


In [0]:
%sql
select measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
where region = 'Northeast';

measure(count)
0


In [0]:
%sql
select age, count(*) as count 
from
customer_pii_data
GROUP BY age;

age,count
33,4
43,8


In [0]:
%sql
select age, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by age

age,measure(count)
33,4
43,8


In [0]:
%sql
select sex, count(*) as count 
from
customer_pii_data
GROUP BY sex;

sex,count
F,2
M,10


In [0]:
%sql
select sex, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by sex

sex,measure(count)
F,2
M,10


In [0]:
%sql
SHOW GROUPS WITH USER `dipankar.kushari@databricks.com`;

name,directGroup
account users,true
dkushari-test-group-2025,true
analyst_usa_tta,true
demo_fgac_group_2,true
team.fieldeng-all,true
mars_group_2,true
vibe-coding,true
admins,true
ANALYST_SPAIN,true


In [0]:
%sql
select is_account_group_member("ANALYST_SPAIN")

"is_account_group_member(""ANALYST_SPAIN"")"
true


In [0]:
%sql
select * from customer_pii_data order by record_id;

record_id,first_name,last_name,date_of_birth,age,sex,address,ssn,region
0,Jane,Brown,1980-01-01,43,M,789 Pine St,222-22-2222,West
0,Jane,Brown,1980-01-01,43,M,789 Pine St,222-22-2222,West
2,John,Johnson,2000-01-01,23,F,789 Pine St,222-22-2222,West
2,John,Johnson,2000-01-01,23,F,789 Pine St,222-22-2222,West
3,John,Johnson,1990-01-01,33,F,456 Elm St,333-33-3333,South
3,John,Johnson,1990-01-01,33,F,456 Elm St,333-33-3333,South
4,John,Williams,2000-01-01,23,F,123 Main St,333-33-3333,South
4,John,Williams,2000-01-01,23,F,123 Main St,333-33-3333,South
5,Jane,Smith,1980-01-01,43,M,123 Main St,111-11-1111,South
5,Jane,Smith,1980-01-01,43,M,123 Main St,111-11-1111,South


In [0]:
%sql
SELECT region, COUNT(*) AS total_customers
FROM customer_pii_data
GROUP BY region;

region,total_customers
Midwest,24
South,26
West,14
Northeast,26


In [0]:
%sql
select count(*) from customer_pii_data;

count(*)
90


In [0]:
%sql
select region, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by region

region,measure(count)
Midwest,24
South,26
West,14
Northeast,26


In [0]:
%sql
select measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
where region = 'Northeast';

measure(count)
26


In [0]:
%sql
select age, count(*) as count 
from
customer_pii_data
GROUP BY age;

age,count
38,6
23,28
33,26
43,22
28,8


In [0]:
%sql
select age, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by age

age,measure(count)
38,6
23,28
33,26
43,22
28,8


In [0]:
%sql
select sex, count(*) as count 
from
customer_pii_data
GROUP BY sex;

sex,count
F,58
M,32


In [0]:
%sql
select sex, measure(count)
from   dkushari_uc.fgac.customer_pii_data_metric_view
group by sex

sex,measure(count)
F,58
M,32
